<a href="https://colab.research.google.com/github/VimalViv/AdvancedSGDS_Group_Project/blob/main/Copy_of_Super_Filter_v2B_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Step 1. Install dependencies

!pip install torch torchvision opencv-python pytesseract pillow tqdm
!pip install open_clip_torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.9 MB/s eta 0:00:00


In [ ]:
# Step 2. Imports + Setup
import os
import cv2
import torch
import shutil
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import pytesseract
import open_clip

from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader

In [ ]:
# Step 3. User Inputs (EDIT THESE)

# Vimal you may need to update this code so that it links to content in your folders correctly?
# Not sure, just flagging

from google.colab import drive
drive.mount('/content/drive')

IMAGE_FOLDER = "/content/drive/MyDrive/1-UCL_Classes/Advanced_SGDS/Preservation_Final_Project/Images/Test_Images" # your input folder
OUTPUT_FOLDER = "/content/drive/MyDrive/1-UCL_Classes/Advanced_SGDS/Preservation_Final_Project/Images/Filter_Output"
NO_IMAGE_PATH = "/content/drive/MyDrive/1-UCL_Classes/Advanced_SGDS/Preservation_Final_Project/Images/invalid_image.jpg"
# your "no image returned" reference

BATCH_SIZE = 4  # adjust (4–8 ideal for Colab GPU)

# Brightness thresholds
DARK_THRESHOLD = 30
BRIGHT_THRESHOLD = 220

Mounted at /content/drive


In [ ]:
# Sanity Check

print("Images folder exists:", os.path.exists(IMAGE_FOLDER))
print("Output folder exists:", os.path.exists(OUTPUT_FOLDER))
print("No-image file exists:", os.path.exists(NO_IMAGE_PATH))

print("Number of images:", len(os.listdir(IMAGE_FOLDER)))

Images folder exists: True
Output folder exists: False
No-image file exists: True
Number of images: 273


In [ ]:
# Step 4. Load segmentation model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.segmentation.deeplabv3_resnet101(pretrained=True)
model.to(device)
model.eval()

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DeepLabV3_ResNet101_Weights.COCO_WITH_VOC_LABELS_V1`. You can also use `weights=DeepLabV3_ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/deeplabv3_resnet101_coco-586e9e4e.pth" to /root/.cache/torch/hub/checkpoints/deeplabv3_resnet101_coco-586e9e4e.pth


100%|██████████| 233M/233M [00:01<00:00, 139MB/s]


DeepLabV3(
  (backbone): IntermediateLayerGetter(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Se

In [ ]:
# Step 5. Image transforms

# this is set at 512x512, not 224x224 because it is better for semantic segmentation

transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
])

In [ ]:
# Step 6. Dataset + DataLoader (batching happens here)

class ImageDataset(Dataset):
    def __init__(self, folder):
        self.paths = [os.path.join(folder, f) for f in os.listdir(folder) if f.endswith(".jpg")]

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        image = Image.open(path).convert("RGB")
        return transform(image), path


dataset = ImageDataset(IMAGE_FOLDER)

dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [ ]:
# Step 7A. Helper functions - Brightness

# Vimal, not sure we need this at all, maybe cut?

def get_brightness(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return np.mean(gray)

In [ ]:
# Step 7B. Blur detection

def get_blur_score(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var()

In [ ]:
# Step 7C. Google logo (OCR)

# changed to template matching instead of OCR, per Chat recommendation should be better

google_template = cv2.imread("/path/to/google_logo_template.jpg", 0)

def has_google_logo(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape

    crop = gray[int(h*0.85):h, 0:int(w*0.3)]

    res = cv2.matchTemplate(crop, google_template, cv2.TM_CCOEFF_NORMED)
    _, max_val, _, _ = cv2.minMaxLoc(res)

    return max_val > 0.6

In [ ]:
# move to top with rest of installs
!pip install ImageHash

In [ ]:
# Step 7D. No image returned (exact match)

# previously set to a pixel-perfect match, need to adjust down
# have it look for near-identical images

import imagehash
from PIL import Image

ref_hash = imagehash.average_hash(Image.open(NO_IMAGE_PATH))

def is_no_image(img_path):
    try:
        img_hash = imagehash.average_hash(Image.open(img_path))
        return abs(img_hash - ref_hash) < 5   # tolerance
    except:
        return False

In [ ]:
# Step 7E. Segmentation ratios

# removing the sky, vegetation, and vehicle filters, only checking that there are enough buildings

# Pascal VOC class IDs used by DeepLab:

def get_building_ratio(pred_mask):
    BUILDING_CLASS_ID = 2  # Pascal VOC

    total = pred_mask.size
    building_pixels = np.sum(pred_mask == BUILDING_CLASS_ID)

    return building_pixels / total

In [ ]:
# Step 8a. Initialize CLIP model for zero-shot

# confirm that this is best CLIP model, had gotten a note from ChatGPT about updating to a different training set?
# possibly mixing up directions here, but noting just in case

# also getting a token error, but there shouldn't even be a token here? unclear

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="openai"
)
clip_model = clip_model.to(device)
clip_model.eval()

tokenizer = open_clip.get_tokenizer("ViT-B-32")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


In [ ]:
# 8b. Define labels
labels = {
    "night": [
        "a street at night",
        "a dark nighttime street",
        "a street with artificial lighting at night"
    ],
    "day": [
        "a street during the day",
        "a bright daytime street"
    ],
    "indoor": [
        "an indoor scene",
        "inside a building"
    ],
    "outdoor": [
        "an outdoor street",
        "a road outside"
    ]
}

In [ ]:
# 8c. Precompute text embeddings

text_features = {}

with torch.no_grad():
    for label, prompts in labels.items():
        tokens = tokenizer(prompts).to(device)
        embeddings = clip_model.encode_text(tokens)
        embeddings /= embeddings.norm(dim=-1, keepdim=True)

        text_features[label] = embeddings.mean(dim=0)

In [ ]:
# Step 8. Create output folders

# removing sky, trees, and vehicles

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

folders = [
    "too_dark", "too_bright", "blurry",
    "no_image", "no_google",
    "no_buildings", "night", "indoor"
]

for f in folders:
    os.makedirs(os.path.join(OUTPUT_FOLDER, f), exist_ok=True)

In [ ]:
# Step 9. MAIN LOOP (batched processing)

# an error is coming up on the google logo search, needs to be resolved

results = []

with torch.no_grad():
    for batch_imgs, batch_paths in tqdm(dataloader):

        # --- Segmentation batch ---
        batch_imgs = batch_imgs.to(device)
        outputs = model(batch_imgs)['out']
        preds = torch.argmax(outputs, dim=1).cpu().numpy()

        for i in range(len(batch_paths)):
            path = batch_paths[i]
            filename = os.path.basename(path)

            img = cv2.imread(path)
            pred_mask = preds[i]

            # --- Metrics ---
            brightness = get_brightness(img)
            blur_score = get_blur_score(img)
            google_logo = has_google_logo(img)
            no_image_flag = is_no_image(path)  # assuming you updated this to hashing

            # --- Building ratio ONLY ---
            building_ratio = get_building_ratio(pred_mask)

            # --- CLIP classification ---
            pil_img = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            clip_input = clip_preprocess(pil_img).unsqueeze(0).to(device)

            image_features = clip_model.encode_image(clip_input)
            image_features /= image_features.norm(dim=-1, keepdim=True)

            scores = {}
            for label, text_feat in text_features.items():
                similarity = (image_features @ text_feat).item()
                scores[label] = similarity

            day_night = max(["day", "night"], key=lambda x: scores[x])
            indoor_outdoor = max(["indoor", "outdoor"], key=lambda x: scores[x])

            # --- Flags ---
            flags = []

            # --- Brightness ---
            if brightness < DARK_THRESHOLD:
                flags.append("too_dark")
                shutil.copy(path, os.path.join(OUTPUT_FOLDER, "too_dark", filename))

            if brightness > BRIGHT_THRESHOLD:
                flags.append("too_bright")
                shutil.copy(path, os.path.join(OUTPUT_FOLDER, "too_bright", filename))

            # --- Blur ---
            if blur_score < 100:
                flags.append("blurry")
                shutil.copy(path, os.path.join(OUTPUT_FOLDER, "blurry", filename))

            # --- Google logo ---
            if not google_logo:
                flags.append("no_google")
                shutil.copy(path, os.path.join(OUTPUT_FOLDER, "no_google", filename))

            # --- Invalid image ---
            if no_image_flag:
                flags.append("no_image")
                shutil.copy(path, os.path.join(OUTPUT_FOLDER, "no_image", filename))

            # --- Building filter (IMPROVED LOGIC) ---
            if (
                building_ratio < 0.15
                and blur_score > 100
                and not no_image_flag
            ):
                flags.append("no_buildings")
                shutil.copy(path, os.path.join(OUTPUT_FOLDER, "no_buildings", filename))

            # --- CLIP: Night ---
            if scores["night"] > scores["day"]:
                flags.append("night")
                shutil.copy(path, os.path.join(OUTPUT_FOLDER, "night", filename))

            # --- CLIP: Indoor (with confidence threshold) ---
            if scores["indoor"] > scores["outdoor"] + 0.05:
                # also guard against garbage images
                if not no_image_flag and blur_score > 100:
                    flags.append("indoor")
                    shutil.copy(path, os.path.join(OUTPUT_FOLDER, "indoor", filename))

            # --- Save result ---
            results.append({
                "image": filename,
                "brightness": brightness,
                "blur_score": blur_score,
                "google_logo": google_logo,
                "no_image": no_image_flag,

                "building_ratio": building_ratio,

                "day_night": day_night,
                "indoor_outdoor": indoor_outdoor,
                "day_score": scores["day"],
                "night_score": scores["night"],
                "indoor_score": scores["indoor"],
                "outdoor_score": scores["outdoor"],

                "flags": ",".join(flags)
            })

  0%|          | 0/69 [00:02<?, ?it/s]


error: OpenCV(4.13.0) /io/opencv/modules/imgproc/src/templmatch.cpp:587: error: (-215:Assertion failed) corr.rows <= img.rows + templ.rows - 1 && corr.cols <= img.cols + templ.cols - 1 in function 'crossCorr'


In [ ]:
# Step 10. Save CSV
df = pd.DataFrame(results)
df.to_csv(os.path.join(OUTPUT_FOLDER, "results_v2.csv"), index=False)

print("✅ Done! Results saved.")

✅ Done! Results saved.
